In [2]:
PREFIX='https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main'

!wget ${PREFIX}/01-agentic-rag/code/ingest.py
!wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
!wget ${PREFIX}/04-evaluation/code/evaluation_utils.py  

://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py: Scheme missing.
://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py: Scheme missing.
://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py: Scheme missing.


In [3]:
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [6]:
documents = documents_llm

In [7]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
response.output_parsed.questions

['I just found this course late — am I still able to join now?',
 'Can I enroll after the course has already started, or is it too late?',
 'If I join now, do I still get access even though I missed the start date?',
 'Is it okay to start the course halfway through, and what about the certificate?',
 "What do I need to do to be eligible for a certificate if I'm joining late?"]

In [16]:
from evaluation_utils import llm_structured
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course—can I still join it now?', 'Am I too late to start, or is it still open for new students?', 'Can I enroll after the course has already started?', 'Is it okay to join the course late and still participate?', 'If I join now, will I still be able to get a certificate?']


In [17]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=82, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=289)

In [18]:
from evaluation_utils import calc_price

In [19]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.000369, 'total_cost': 0.00052425}

In [20]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course—can I still join it now?',
  'document': '74eb249bbf'},
 {'question': 'Am I too late to start, or is it still open for new students?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to join the course late and still participate?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, will I still be able to get a certificate?',
  'document': '74eb249bbf'}]

In [21]:
import pandas as pd
pd.DataFrame(records)

,question,document
0,I just found this course—can I still join it now?,74eb249bbf
1,"Am I too late to start, or is it still open fo...",74eb249bbf
2,Can I enroll after the course has already star...,74eb249bbf
3,Is it okay to join the course late and still p...,74eb249bbf
4,"If I join now, will I still be able to get a c...",74eb249bbf


In [22]:
from evaluation_utils import llm_structured_retry


In [23]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [24]:
generate_ground_truth(doc)

([{'question': 'I just found this course — is it still okay to join now, or am I too late?',
   'document': '74eb249bbf'},
  {'question': 'Can I enroll after the course has already started and still participate?',
   'document': '74eb249bbf'},
  {'question': 'If I join late, can I still get a certificate somehow?',
   'document': '74eb249bbf'},
  {'question': 'What’s the deadline for the project if I want to be eligible for the certificate?',
   'document': '74eb249bbf'},
  {'question': 'Is it fine to start the course now, even if submissions are still open?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=302))

In [25]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [26]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [27]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [28]:
results[1]

([{'question': 'I signed up for the LLM Zoomcamp — when should I get the confirmation email, or am I missing something?',
   'document': '977bf7786c'},
  {'question': 'Do I actually need a registration confirmation to join the course and submit homework?',
   'document': '977bf7786c'},
  {'question': 'If I never got an email after registering, can I still start the LLM Zoomcamp anyway?',
   'document': '977bf7786c'},
  {'question': 'Is there a real acceptance step for the course, or does registration just help you track interest?',
   'document': '977bf7786c'},
  {'question': 'Do I need to be on some registered list before I can do the homework, or is that not required?',
   'document': '977bf7786c'}],
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=116, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=354))

In [29]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [30]:
ground_truth[0]

{'question': 'I just found this course late — can I still join, or is it too late to start?',
 'document': '74eb249bbf'}

In [31]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07706325

In [32]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07706325

In [33]:
df_ground_truth = pd.DataFrame(ground_truth)

In [34]:
df_ground_truth

,question,document
0,I just found this course late — can I still jo...,74eb249bbf
1,Am I allowed to enroll if I only discovered th...,74eb249bbf
2,"If I join after the course has started, can I ...",74eb249bbf
3,What do I need to do to qualify for the certif...,74eb249bbf
4,"Is it okay to start the course now, and what’s...",74eb249bbf
...,...,...
510,Why does pip keep giving me requests 2.28 when...,4b30b918bc
511,Is there a way to install requests straight fr...,4b30b918bc
512,I’m getting a 401 Client Error during this set...,4b30b918bc
513,What’s the exact pip command to grab the newer...,4b30b918bc


In [35]:
import os

os.makedirs("data", exist_ok=True)
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [36]:
len(df_ground_truth)

515